In [ ]:
import pandas as pd

from slrrag.settings import PROJECT_ROOT
from slrrag.toolbox.files import slugify

%load_ext autoreload
%autoreload 2
PATH_JSON_ZOTERO = PROJECT_ROOT / "data" / "temp" / "Exported Items.json"
PATH_JSON_ZOTERO.exists(), PATH_JSON_ZOTERO

In [ ]:
import json

with open(PATH_JSON_ZOTERO, "r") as file_json_zotero:
    json_zotero = json.load(file_json_zotero)

df_zotero = pd.DataFrame.from_dict(json_zotero["items"])
df_zotero["doi"] = df_zotero["DOI"].str.lower()
df_zotero["title_slug"] = df_zotero["title"].apply(slugify)

df_zotero.head().T

In [ ]:
df_zotero.query("url.isna()")

In [ ]:
df_zotero[["title", "doi"]]

In [ ]:
df_zotero[df_zotero["title"].str.contains("Visualization")].T

In [ ]:
PATH_CSV_OBSIDIAN_ON_ZOTERO = PROJECT_ROOT / "data" / "temp" / "On zotero.csv"
df_obsidian = pd.read_csv(PATH_CSV_OBSIDIAN_ON_ZOTERO)  # .query("tags == '#keep'")
df_obsidian["doi"] = df_obsidian["doi"].str.lower()
df_obsidian["title_slug"] = df_obsidian["title"].apply(slugify)
df_obsidian

In [ ]:
df_obsidian["title_merge"] = df_obsidian["title_slug"].apply(slugify)
df_zotero["title_merge"] = df_zotero["title_slug"].apply(slugify)

df_m = pd.merge(
    df_obsidian, df_zotero, on="title_merge", how="outer", suffixes=("_obs", "_zot")
)  # [["doi_zot","doi_obs","title_slug","title_obs","title_zot","accessed"]]
df_m

In [ ]:
df_m[df_m["title_slug_obs"].isna()][["title_zot", "title_slug_zot"]]

In [ ]:
df_m[df_m["title_slug_zot"].isna()][["title_obs", "title_slug_obs"]]

## Test with downloading PDF from Open alex

In [ ]:
import os
from pathlib import Path

import pyalex
import requests
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")
# def download_pdf_openalex(pdf_link:str):
pyalex.config.api_key = os.environ["OPENALEX_API"]
# download_pdf_openalex(df_obsidian["pdf"].values)
tmp = "https://content.openalex.org/works/w2177083728.pdf"

In [ ]:
PATH_OUT_TMP = PROJECT_ROOT / "out.pdf"


def download_pdf_openalex(url_openalex: str, path_out: Path):
    url_with_key = url_openalex + "?api_key=" + os.environ["OPENALEX_API"]
    with requests.get(url_with_key, stream=True) as r:
        r.raise_for_status()
        with open(path_out, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                # If you have chunk encoded response uncomment if
                # and set chunk_size parameter to None.
                # if chunk:
                f.write(chunk)


download_pdf_openalex(tmp, PATH_OUT_TMP)
tmp

In [ ]:
from tqdm import tqdm

from slrrag.settings import PATH_OUTPUT

PATH_OUT_PDF = PATH_OUTPUT / "pdf_keep"
PATH_OUT_PDF.mkdir(exist_ok=True, parents=True)
for r in tqdm(df_obsidian.itertuples()):
    download_pdf_openalex(r.pdf.lower(), PATH_OUT_PDF / r.title)

In [ ]:
df_obsidian.query("accessed.isna() | accessed.str.contains('Yes')").groupby(
    "title_slug"
).size().sort_values(ascending=False)

In [ ]:
df_zotero.groupby("title_slug").size().sort_values(ascending=False)

In [ ]:
df_m[["title_merge", "title_obs", "title_zot"]].query(
    "title_obs.isna() | title_zot.isna()"
).sort_values("title_merge")

In [ ]:
df_m["title_slug"]

In [ ]:
df_m = pd.merge(
    df_obsidian.query("doi.isna()"),
    df_zotero.query("doi.isna()"),
    on="title_slug",
    how="outer",
    suffixes=("_obs", "_zot"),
)[["doi_zot", "doi_obs", "title_slug", "title_obs", "title_zot", "accessed"]]
df_m.sort_values("title_slug")